# Phase 6: Business evaluation

This notebook evaluates the same 72 bilingual business questions with a base model, a LoRA checkpoint, and LoRA plus adaptive sampling. It does not invent a score when a checkpoint is missing.

In [ ]:
from pathlib import Path
import json
import sys
from collections import Counter

_candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents, Path('/kaggle/working/SLM-PRJ')]
ROOT = next((p for p in _candidates if (p / 'pyproject.toml').is_file() and (p / 'benchmarks').is_dir()), Path('/kaggle/working/SLM-PRJ'))
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from business.create_business_db import create_database, verify_questions

DB_PATH = ROOT / 'business' / 'business.sqlite'
QUESTIONS_PATH = ROOT / 'business' / 'business_questions.json'
create_database(DB_PATH, QUESTIONS_PATH)
print(verify_questions(DB_PATH, QUESTIONS_PATH))
questions = json.loads(QUESTIONS_PATH.read_text(encoding='utf-8'))
len(questions), Counter(item['difficulty'] for item in questions)

In [ ]:
from evaluation.final_eval import evaluate_pipeline_records, write_json
from core.config import load_config

# Set these paths on Kaggle before enabling RUN_EVAL.
import os
MODEL_PATHS = {
    'base': Path(os.environ['BUSINESS_BASE_MODEL']) if os.environ.get('BUSINESS_BASE_MODEL') else None,
    'lora': Path(os.environ['BUSINESS_LORA_MODEL']) if os.environ.get('BUSINESS_LORA_MODEL') else None,
}
RUN_EVAL = False
SEEDS = [42]

def variant_config(model_path: Path, mode: str, seed: int) -> dict:
    config = load_config()
    config['model']['backend'] = 'huggingface'
    config['model']['model_id'] = str(model_path)
    config['model']['local_files_only'] = True
    config['model']['device'] = 'cuda'
    config['model']['dtype'] = 'float16'
    config['model']['quantization'] = 'none'
    config['seed'] = seed
    config['sampling']['mode'] = mode
    config['sampling']['fixed_k'] = 16
    config['sampling']['max_candidates'] = 32
    config['sampling']['rounds'] = [4, 4, 8, 16]
    return config

def business_db_path(db_id: str) -> Path:
    if db_id != 'business':
        raise ValueError(f'Unexpected business db_id: {db_id}')
    return DB_PATH

reports = {}
if RUN_EVAL:
    for seed in SEEDS:
        for variant, mode in [('base', 'greedy'), ('lora', 'greedy'), ('lora_adaptive', 'adaptive_sc')]:
            model_path = MODEL_PATHS['base' if variant == 'base' else 'lora']
            if model_path is None or not model_path.exists():
                print(f'SKIP {variant} seed={seed}: model path missing: {model_path}')
                continue
            report_path = ROOT / 'results' / 'business' / f'{variant}_seed{seed}.json'
            reports[f'{variant}_seed{seed}'] = evaluate_pipeline_records(
                questions, business_db_path, variant_config(model_path, mode, seed),
                variant_name=variant, output_path=report_path,
            )
else:
    print('RUN_EVAL=False: no model score is reported.')

In [ ]:
def error_table(report: dict) -> list[dict]:
    failed = [item for item in report.get('records', []) if not item['correct']]
    counts = Counter(item['error_type'] or 'unknown' for item in failed)
    return [{'error_type': key, 'count': value} for key, value in sorted(counts.items())]

for name, report in reports.items():
    print(name, {
        'accuracy': report['execution_accuracy'],
        'questions': report['questions'],
        'average_candidates': report['average_candidates'],
        'errors': error_table(report),
    })

In [ ]:
# Gradio demo: question -> SQL -> result -> consensus.
import os
from core.pipeline import Pipeline

_demo_pipeline = None
_demo_variant = 'lora_adaptive'

def _get_demo_pipeline(variant: str):
    global _demo_pipeline, _demo_variant
    if _demo_pipeline is not None and _demo_variant == variant:
        return _demo_pipeline
    key = 'base' if variant == 'base' else 'lora'
    path = MODEL_PATHS.get(key)
    if path is None or not path.exists():
        raise FileNotFoundError(f'Model path missing for {variant}: {path}')
    mode = 'adaptive_sc' if variant == 'lora_adaptive' else 'greedy'
    _demo_pipeline = Pipeline(variant_config(path, mode, 42))
    _demo_variant = variant
    return _demo_pipeline

def predict_business(question: str, variant: str = 'lora_adaptive'):
    pipeline = _get_demo_pipeline(variant)
    result = pipeline.run(DB_PATH, question)
    selected = result.selected
    if selected is None or selected.execution is None or not selected.execution.success:
        return '', [], 'No executable SQL selected', result.explanation
    rows = [dict(zip(selected.execution.columns, row)) for row in selected.execution.rows]
    consensus = f'{selected.vote_score:.1%} of executable candidates; {len(result.candidates)} sampled'
    return selected.sql, rows, consensus, result.explanation

try:
    import gradio as gr
    with gr.Blocks(title='Business Text-to-SQL') as demo:
        gr.Markdown('# Business Text-to-SQL')
        question_box = gr.Textbox(label='Question', lines=3)
        variant_box = gr.Dropdown(['base', 'lora', 'lora_adaptive'], value='lora_adaptive', label='Variant')
        run_button = gr.Button('Run')
        sql_box = gr.Code(label='SQL', language='sql')
        result_box = gr.JSON(label='Result')
        consensus_box = gr.Textbox(label='Consensus')
        explanation_box = gr.Textbox(label='Explanation')
        run_button.click(predict_business, [question_box, variant_box], [sql_box, result_box, consensus_box, explanation_box])
    demo
except (ImportError, AttributeError):
    print('Gradio is optional and is not installed in this environment.')